## Grid Task

In [ ]:
%load_ext autoreload
%autoreload 2

from fibsem import utils, acquire
from fibsem.structures import BeamType, ImageSettings

microscope, settings = utils.setup_session()

EXP_PATH = r"C:\Users\Admin\Documents\GitHub\openfibsem\fibsem\fibsem\applications\autolamella\log\_\AutoLamella-2026-05-06-12-10-DEV"
SAMPLE_HOLDER_PATH = r"C:\Users\Admin\Documents\GitHub\openfibsem\fibsem\fibsem\config\leica-sample-holder.yaml"

In [ ]:
from fibsem.microscopes._stage import SampleGrid, SampleHolder, GridSlot
sh = SampleHolder.load(SAMPLE_HOLDER_PATH)
from pprint import pprint

microscope._stage.holder = sh
pprint(microscope._stage.holder.to_dict())



In [ ]:
microscope._stage.holder.slots["Slot-01"].loaded_grid
microscope._stage.holder.slots["Slot-02"].loaded_grid

In [ ]:
microscope._stage.move_to_slot("Slot-01")

microscope.acquire_image(beam_type=BeamType.ELECTRON)
microscope.acquire_image(beam_type=BeamType.ION)

In [ ]:
from fibsem.applications.autolamella.workflows.tasks.grid_tasks import GridTask, AcquireOverviewImageGridTask, AcquireOverviewImageGridTaskConfig
from fibsem.applications.autolamella.structures import Experiment
import os 
exp = Experiment.load(os.path.join(EXP_PATH, "experiment.yaml"))



In [ ]:
grid =  microscope._stage.holder.slots["Slot-01"].loaded_grid

config = AcquireOverviewImageGridTaskConfig(task_name="Acquire SEM Overview", orientation="SEM")
task =  AcquireOverviewImageGridTask(microscope, config, grid=grid, experiment=exp)
# task.run()

In [ ]:
from fibsem.applications.autolamella.workflows.tasks.grid_tasks import run_grid_task, run_grid_tasks, AcquireImageGridTaskConfig

run_grid_tasks(
    microscope=microscope,
    experiment=exp,
    grid_names=["Grid-01", "Grid-02"],
    task_names=[AcquireImageGridTaskConfig.task_type, AcquireOverviewImageGridTaskConfig.task_type]
)

# High V Imaging
# GIS Deposition



## GIS Deposition Workflow

1. go to grid centre
2. coincident position
3. lower ~1-2mm -> 'safe position' ( go to deposition position, maybe rotation/tilt)
4. move back to centre
5. insert gis
6. draw line, zoom 30um
7. set pattern time 30s
7a  alt: open gis, set timer
8. retract gis
9. restore initial position

In [ ]:

microscope._stage.move_to_slot("Slot-02")


In [ ]:
from fibsem.microscopes.autoscript import AutoscriptGISPort

gis_port = AutoscriptGISPort(microscope)

print(gis_port.port_name)
print(gis_port._port)

gis_port._move_to_safe_gis_position()
gis_port._run_safety_check()

input()
gis_port.run_deposition(10)

In [ ]:
print(microscope.get_stage_position().z*1e3)